In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
import glob
import os
import regex as re
import csv

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# SEIRS-SEI Manaus Model - Population Correction

This notebook corrects the scale mismatch between model population and observed cases.

So far, the project has considered analyzing rural cases in the municipality of Manaus. Given that there is no data based only on this zone, we use a formula derived from the Trajetórias Project (https://www.nature.com/articles/s41597-023-01962-1):

$$\text{Incidence}(d,m,z,t_1,t_2)=\dfrac{\text{Cases}(d,m,z,t_1,t_2)}{\text{Pop}(m,z,(t_1+t_2)/2)\times 5\text{ years}}\times 10^5$$

The parameters are:
<ol>
  <p>$d$: disease (Chagas, CL, VL, Dengue, Falciparum, Vivax, Vivax+Falciparum)</p>
  <p>$m$: municipality</p>
  <p>$z$: zone (rural, urban or total)</p>
  <p>$[t_1,t_2]$: [2004,2008] or [2015, 2019]</p>
</ol>

<!-- ## Problem
- Model was using rural population (~8,558)
- Cases represent ALL of Manaus (~2M population)
- This caused inconsistency in model-data comparison

## Solution Options
1. **Option A**: Scale observed cases to rural level (multiply by 0.0037)
2. **Option B**: Use full municipal population in model (~2M)

We implement both approaches for comparison. -->

Below, we will approximate the population and number of cases in 2020, as the mean of our period [2017,2023], based on extrapolation of known data.

## Load Data

In [5]:
DATA_DIR = '../../data_files/data'

climate_data = pd.read_csv(DATA_DIR + '/climate_api_data_2016_2024.csv')
cases_data = pd.read_csv(DATA_DIR + '/sivep_notification_data/treated_malaria_notification_data/cumulative_manaus_cases_2016_2023.csv')
pop_data = pd.read_csv(DATA_DIR + '/ibge_manaus_population_data_2016_2024.csv')
defor_data = pd.read_csv(DATA_DIR + '/deter_notification_data/treated_deter_deforestation_data_2016_2024.csv')
fires_data = pd.read_csv(DATA_DIR + '/inpe_fire_counts_data_2016_2024.csv')

climate_data['date'] = pd.to_datetime(climate_data['date'])
cases_data['date'] = pd.to_datetime(cases_data['date'])
pop_data['date'] = pd.to_datetime(pop_data['index'])
defor_data['date'] = pd.to_datetime(defor_data['date'])
fires_data['date'] = pd.to_datetime(fires_data['date'])

# Filter to 2017-2023
start_date = pd.to_datetime('2017-01-01')
end_date = pd.to_datetime('2023-12-31')
climate_data = climate_data[(climate_data['date'] >= start_date) & (climate_data['date'] <= end_date)].reset_index(drop=True)
cases_data = cases_data[(cases_data['date'] >= start_date) & (cases_data['date'] <= end_date)].reset_index(drop=True)
pop_data = pop_data[(pop_data['date'] >= start_date) & (pop_data['date'] <= end_date)].reset_index(drop=True)
defor_data = defor_data[(defor_data['date'] >= start_date) & (defor_data['date'] <= end_date)].reset_index(drop=True)
fires_data = fires_data[(fires_data['date'] >= start_date) & (fires_data['date'] <= end_date)].reset_index(drop=True)

print(f"Climate data: {len(climate_data)} days")
print(f"Cases data: {len(cases_data)} days")
print(f"Population data: {len(pop_data)} days")
print(f"Deforestation data: {len(defor_data)} days")
print(f"Fires data: {len(fires_data)} days")

Climate data: 2556 days
Cases data: 2556 days
Population data: 2556 days
Deforestation data: 2556 days
Fires data: 2556 days


In [6]:
with open("../../data_files/data/TRAJETORIAS_DATASET_Epidemiological_dimension_indicators.csv", encoding="utf-8") as f:
    lines = f.readlines()

clean_lines = [line.replace('"', '') for line in lines]

trajetorias_epidem_df = pd.DataFrame([line.strip().split(",") for line in clean_lines])
trajetorias_epidem_df.columns = trajetorias_epidem_df.iloc[0]
trajetorias_epidem_df = trajetorias_epidem_df[1:].reset_index(drop=True)
trajetorias_epidem_df.columns = trajetorias_epidem_df.columns.str.replace('\ufeff', '')

with open("../../data_files/data/TRAJETORIAS_DATASET_Population_indicators.csv", encoding="utf-8") as f:
    lines = f.readlines()

clean_lines = [line.replace('"', '') for line in lines]

trajetorias_pop_df = pd.DataFrame([line.strip().split(",") for line in clean_lines])
trajetorias_pop_df.columns = trajetorias_pop_df.iloc[0]
trajetorias_pop_df = trajetorias_pop_df[1:].reset_index(drop=True)
trajetorias_pop_df.columns = trajetorias_pop_df.columns.str.replace('\ufeff', '')

manaus_epidem_df = trajetorias_epidem_df[trajetorias_epidem_df['municipality']=='Manaus']
manaus_epidem_df = manaus_epidem_df.reset_index(drop=True)
manaus_vivax_df = manaus_epidem_df[manaus_epidem_df['disease']=='Vivax']
manaus_vivax_df = manaus_vivax_df.reset_index(drop=True)
manaus_vivax_df = manaus_vivax_df.drop(columns=['state_abbrev', 'state', 'municipality', 'geocode', 'disease'])

cols = ['cases', 'inc']

manaus_vivax_df[cols] = manaus_vivax_df[cols].apply(pd.to_numeric, errors='coerce')

manaus_pop_df = trajetorias_pop_df[trajetorias_pop_df['municipality']=='Manaus']
manaus_pop_df = manaus_pop_df.reset_index(drop=True)
manaus_rural_tot_pop_df = manaus_pop_df.drop(columns=['state_abbrev', 'state', 'municipality', 'geocode', 'prop_urb2000', 'prop_urb2010'])

cols = ['urb2000', 'rur2000', 'tot2000', 'prop_rur2000',
    'urb2010', 'rur2010', 'tot2010', 'prop_rur2010',
    'pop_estimated2006', 'pop_estimated2017',
    'urb2006e', 'rur2006e', 'urb2017e', 'rur2017e']

manaus_rural_tot_pop_df[cols] = manaus_rural_tot_pop_df[cols].apply(pd.to_numeric, errors='coerce')

In [7]:
manaus_vivax_df

,period,zone,cases,inc
0,2004-2008,rural,78745,184030.772087
1,2015-2019,rural,25859,60433.700367
2,2004-2008,urban,183519,2184.793966
3,2015-2019,urban,36387,433.187289
4,2004-2008,total,262264,3106.429047
5,2015-2019,total,62246,584.397051


In [8]:
manaus_rural_tot_pop_df

,urb2000,rur2000,tot2000,prop_rur2000,urb2010,rur2010,tot2010,prop_rur2010,pop_estimated2006,pop_estimated2017,urb2006e,rur2006e,urb2017e,rur2017e
0,1396768,9067,1405835,0.00645,1792881,9133,1802014,0.005068,1688524,2130264,1.679966e+06,8557.807926,2.119467e+06,10796.642597


In [9]:
# Calculate exact ratios from TRAJETORIAS data
print("=== Calculated from TRAJETORIAS Dataset ===\n")

# Filter data for each period and zone
rural_04_08 = manaus_vivax_df[(manaus_vivax_df['period']=='2004-2008') & (manaus_vivax_df['zone']=='rural')]
total_04_08 = manaus_vivax_df[(manaus_vivax_df['period']=='2004-2008') & (manaus_vivax_df['zone']=='total')]
rural_15_19 = manaus_vivax_df[(manaus_vivax_df['period']=='2015-2019') & (manaus_vivax_df['zone']=='rural')]
total_15_19 = manaus_vivax_df[(manaus_vivax_df['period']=='2015-2019') & (manaus_vivax_df['zone']=='total')]

rural_cases_04_08 = rural_04_08['cases'].values[0]
total_cases_04_08 = total_04_08['cases'].values[0]
rural_cases_15_19 = rural_15_19['cases'].values[0]
total_cases_15_19 = total_15_19['cases'].values[0]

rural_inc_04_08 = rural_04_08['inc'].values[0]
total_inc_04_08 = total_04_08['inc'].values[0]
rural_inc_15_19 = rural_15_19['inc'].values[0]
total_inc_15_19 = total_15_19['inc'].values[0]

# Case ratios
case_ratio_04_08 = rural_cases_04_08 / total_cases_04_08
case_ratio_15_19 = rural_cases_15_19 / total_cases_15_19

print(f"Case Ratios (Rural/Total):")
print(f"  2004-2008: {case_ratio_04_08*100:.2f}% ({rural_cases_04_08:,}/{total_cases_04_08:,})")
print(f"  2015-2019: {case_ratio_15_19*100:.2f}% ({rural_cases_15_19:,}/{total_cases_15_19:,})")

=== Calculated from TRAJETORIAS Dataset ===

Case Ratios (Rural/Total):
  2004-2008: 30.03% (78,745/262,264)
  2015-2019: 41.54% (25,859/62,246)


In [10]:
# Population from incidence formula: Pop = (Cases × 100000) / (Inc × 5 years)
pop_rural_04_08 = rural_cases_04_08 * 100000 / (rural_inc_04_08 * 5)
pop_total_04_08 = total_cases_04_08 * 100000 / (total_inc_04_08 * 5)
pop_ratio_04_08 = pop_rural_04_08 / pop_total_04_08

pop_rural_15_19 = rural_cases_15_19 * 100000 / (rural_inc_15_19 * 5)
pop_total_15_19 = total_cases_15_19 * 100000 / (total_inc_15_19 * 5)
pop_ratio_15_19 = pop_rural_15_19 / pop_total_15_19

print(f"\nPopulation Ratios (Rural/Total) from Incidence:")
print(f"  2004-2008: {pop_ratio_04_08*100:.2f}% (rural pop: {pop_rural_04_08:,.0f})")
print(f"  2015-2019: {pop_ratio_15_19*100:.2f}% (rural pop: {pop_rural_15_19:,.0f})")


Population Ratios (Rural/Total) from Incidence:
  2004-2008: 0.51% (rural pop: 8,558)
  2015-2019: 0.40% (rural pop: 8,558)


Now, we would like to see the behavior of the transmission and population in the period of our study, which is 2017 to 2023.

Given that the years of 2017 to 2019 are already included in the estimates above, we should only consider 2020 to 2023.

Let's assume the annual growth rate of the disease is given by

$$r=\left(\dfrac{\text{Inc}_{[2015,2019]}}{\text{Inc}_{[2004,2008]}}\right)^{1/11}-1$$

Then $\text{Inc}_y$, for each year $y$, is given by

$$\text{Inc}_y=\text{Inc}_{[2015,2019]}\times(1+r)^{y-2017} $$

In [14]:
rural_inc_15_19

np.float64(60433.7003670624)

In [15]:
rural_inc_15_19/rural_inc_04_08

np.float64(0.3283891040700998)

In [16]:
r = (rural_inc_15_19/rural_inc_04_08)**(1/11) -1
r

np.float64(-0.09627699051711025)

In [17]:
rural_inc_20 = rural_inc_15_19*(1+r)**(2020-2017)
rural_inc_21 = rural_inc_15_19*(1+r)**(2021-2017)
rural_inc_22 = rural_inc_15_19*(1+r)**(2022-2017)
rural_inc_23 = rural_inc_15_19*(1+r)**(2023-2017)

print(f'The rural incidence in the years of 2020 to 2023 are estimated to be, respectively: \n{rural_inc_20}, {rural_inc_21}, {rural_inc_22}, {rural_inc_23}')

The rural incidence in the years of 2020 to 2023 are estimated to be, respectively: 
44605.1707987237, 40310.7191927209, 36429.72446326541, 32922.38022657467


In [18]:
pop_data_clean = pop_data.copy()
pop_data_clean.drop(columns=['index', 'ADJUSTED_RURAL_POP'])
pop_data_clean = pop_data_clean[['date', 'ADJUSTED_POPULATION']]
pop_data_clean

,date,ADJUSTED_POPULATION
0,2017-01-01,1.954657e+06
1,2017-01-02,1.954717e+06
2,2017-01-03,1.954776e+06
3,2017-01-04,1.954836e+06
4,2017-01-05,1.954896e+06
...,...,...
2551,2023-12-27,2.222621e+06
2552,2023-12-28,2.222840e+06
2553,2023-12-29,2.223059e+06
2554,2023-12-30,2.223279e+06


In [19]:
# ============================================
# EXTRAPOLATE RURAL PROPORTION FOR 2017-2023
# ============================================

# Extract rural proportion from Trajetorias data
# From manaus_rural_tot_pop_df:
# prop_rur2000 = 0.00645 (year 2000)
# prop_rur2010 = 0.005068 (year 2010)
# But we need proportions for 2006 (midpoint 2004-2008) and 2017 (midpoint 2015-2019)

# From the dataset, we have:
# rur2006e = 8,557.8 (estimated rural population in 2006)
# pop_estimated2006 = 1,688,524 (total population in 2006)
# rur2017e = 10,796.64 (estimated rural population in 2017)
# pop_estimated2017 = 2,130,264 (total population in 2017)

rural_pop_2006 = manaus_rural_tot_pop_df['rur2006e'].values[0]
total_pop_2006 = manaus_rural_tot_pop_df['pop_estimated2006'].values[0]
prop_rural_2006 = rural_pop_2006 / total_pop_2006

rural_pop_2017 = manaus_rural_tot_pop_df['rur2017e'].values[0]
total_pop_2017 = manaus_rural_tot_pop_df['pop_estimated2017'].values[0]
prop_rural_2017 = rural_pop_2017 / total_pop_2017

print(f"Rural proportion in 2006: {prop_rural_2006:.6f} ({prop_rural_2006*100:.4f}%)")
print(f"Rural proportion in 2017: {prop_rural_2017:.6f} ({prop_rural_2017*100:.4f}%)")

# Calculate annual change rate for rural proportion (using exponential decay)
# Since rural proportion tends to decrease, exponential model: p(y) = p_2017 * (1 + r_p)^(y-2017)
# Where r_p = (p_2017/p_2006)^(1/11) - 1

r_p = (prop_rural_2017 / prop_rural_2006) ** (1/11) - 1
print(f"\nAnnual change rate for rural proportion: {r_p:.6f} ({r_p*100:.4f}% per year)")

# Extrapolate rural proportion for 2020-2023
years = [2020, 2021, 2022, 2023]
prop_rural_extrap = {}

for y in years:
    prop_rural_extrap[y] = prop_rural_2017 * (1 + r_p) ** (y - 2017)
    print(f"Projected rural proportion for {y}: {prop_rural_extrap[y]:.6f} ({prop_rural_extrap[y]*100:.4f}%)")

Rural proportion in 2006: 0.005068 (0.5068%)
Rural proportion in 2017: 0.005068 (0.5068%)

Annual change rate for rural proportion: 0.000000 (0.0000% per year)
Projected rural proportion for 2020: 0.005068 (0.5068%)
Projected rural proportion for 2021: 0.005068 (0.5068%)
Projected rural proportion for 2022: 0.005068 (0.5068%)
Projected rural proportion for 2023: 0.005068 (0.5068%)


In [34]:
# ============================================
# CALCULATE DAILY RURAL POPULATION
# ============================================

# Create a function to get rural proportion for any date
def get_rural_proportion(date, prop_2006, prop_2017, year_2006=2006, year_2017=2017):
    """
    Calculate rural proportion for a given date using exponential interpolation/extrapolation
    """
    # Convert date to decimal year
    year = date.year
    day_of_year = date.timetuple().tm_yday
    days_in_year = 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365
    year_frac = year + (day_of_year - 0.5) / days_in_year
    
    if year_frac <= year_2017:
        # Interpolate between 2006 and 2017
        t = (year_frac - year_2006) / (year_2017 - year_2006)
        # Linear interpolation on log scale for exponential trend
        prop = prop_2006 * (prop_2017 / prop_2006) ** t
    else:
        # Extrapolate beyond 2017
        r_p = (prop_2017 / prop_2006) ** (1/(year_2017 - year_2006)) - 1
        prop = prop_2017 * (1 + r_p) ** (year_frac - year_2017)
    
    return prop

# Calculate rural proportion for each day in pop_data
pop_data_clean = pop_data[['date', 'ADJUSTED_POPULATION']].copy()
pop_data_clean['rural_proportion'] = pop_data_clean['date'].apply(
    lambda d: get_rural_proportion(d, prop_rural_2006, prop_rural_2017)
)
pop_data_clean['rural_population'] = pop_data_clean['ADJUSTED_POPULATION'] * pop_data_clean['rural_proportion']

print(f"\nRural population statistics (2017-2023):")
print(f"  Min: {pop_data_clean['rural_population'].min():,.0f}")
print(f"  Max: {pop_data_clean['rural_population'].max():,.0f}")
print(f"  Mean: {pop_data_clean['rural_population'].mean():,.0f}")

pop_data_clean


Rural population statistics (2017-2023):
  Min: 9,907
  Max: 11,269
  Mean: 10,377


,date,ADJUSTED_POPULATION,rural_proportion,rural_population
0,2017-01-01,1.954657e+06,0.005068,9906.628018
1,2017-01-02,1.954717e+06,0.005068,9906.930820
2,2017-01-03,1.954776e+06,0.005068,9907.233621
3,2017-01-04,1.954836e+06,0.005068,9907.536423
4,2017-01-05,1.954896e+06,0.005068,9907.839225
...,...,...,...,...
2551,2023-12-27,2.222621e+06,0.005068,11264.727601
2552,2023-12-28,2.222840e+06,0.005068,11265.838637
2553,2023-12-29,2.223059e+06,0.005068,11266.949674
2554,2023-12-30,2.223279e+06,0.005068,11268.060711
